In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import ffmpeg as ff
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
from tqdm import tqdm
import multiprocessing
from cartooner import cartoonize
import pickle
import datetime
from facenet_pytorch import MTCNN
import logging
import torch.nn as nn
from einops import rearrange
import logging
import torch
from torchvision import transforms, utils, models
from PIL import Image
# import face_recognition

In [2]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
# streamHandler = logging.StreamHandler()
# streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
# logger.addHandler(streamHandler)
logger.addHandler(fileHandler)

In [3]:
USE_CUDA = torch.cuda.is_available()
device = torch.device('cuda:0' if USE_CUDA else 'cpu')
number_of_samples =120


In [4]:
def get_number_of_frames(file_path: str) -> int:
    probe = ff.probe(file_path)
    video_streams = [stream for stream in probe["streams"] if stream["codec_type"] == "video"]
    #width = video_streams[0]['coded_width']
    #height = video_streams[0]['coded_height']
    del probe
    return video_streams[0]['nb_frames']

In [ ]:
def extract_audio_waveform(file_path: str, sr: int = 16000) -> np.ndarray:
    (
        ff
        .input(file_path)
        .output('pipe:', format='f32le', acodec='pcm_f32le', ac=1, ar=str(sr), loglevel='quiet')
        .run(capture_stdout=True)
    )
    raw = ff.input(file_path).output('pipe:', format='f32le', acodec='pcm_f32le', ac=1, ar=str(sr), loglevel='quiet').run(capture_stdout=True)[0]
    waveform = np.frombuffer(raw, dtype=np.float32)  
    return waveform

In [ ]:
def extract_train_N_video_frames(frame_num: str ,file_path: str, index: int, number_of_samples: int = 15) -> List[np.ndarray]:
    full_video_frames = []#프레임 데이터를 저장할 배열 만들기
                   # Final image size
    trans=transforms.ToTensor()
    frame_num=number_of_samples #추출된 프레임의 개수
    begin_num=0 # 프레임의 시작index
    indexes=[None] * number_of_samples # [0,2.7,5.4......]->[0,2,5......] // [0, 1, 2, 3, 4, 5.... 450?]
    get_frames=int(get_number_of_frames(file_path))
    x=get_frames/number_of_samples
    
    for i in range(frame_num):
        indexes[i]=int(begin_num)
        # print(type(get_number_of_frames(file_path)))
        begin_num+=x
    cap = cv2.VideoCapture(file_path)
    for ind in indexes:
        i=ind
        cap.set(1, ind)
        res, frame = cap.read()#프레임을 읽습니다
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)           # (H,W,3) uint8
        full_video_frames.append(frame)
    cap.release()
    del cap, indexes
    return full_video_frames
    

In [ ]:
def resize_image(image: np.ndarray, new_size: Tuple[int,int]) -> np.ndarray:
    image = np.array(image)
    return cv2.resize(image, new_size, interpolation = cv2.INTER_AREA)

In [8]:
def crop_image_window(image: np.ndarray, training: bool = True) -> np.ndarray:#这个不是太理解·······························
    height, width, _ = image.shape
    if training:
        MAX_N = height - 128
        MAX_M = width - 128
        rand_N_index, rand_M_index = random.randint(0, MAX_N) , random.randint(0, MAX_M)
        return image[rand_N_index:(rand_N_index+128),rand_M_index:(rand_M_index+128),:]
    else:
        N_index = (height - 128) // 2
        M_index = (width - 128) // 2
        return image[N_index:(N_index+128),M_index:(M_index+128),:]

In [9]:
def reading_label_data(file_name: str, dictionary: Dict[str,str]) -> np.ndarray:#영상의 5대 점수를 획득했습니다
    # features = ['extraversion', 'neuroticism', 'agreeableness', 'conscientiousness', 'openness']
    features = ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']
    extracted_data = [float(dictionary[label][file_name]) for label in features]
    return np.stack(extracted_data).reshape(5,1)

In [10]:
def find_script_by_id(input_df, input_id):
    selected_row = input_df[input_df['id'] == input_id]
    
    if not selected_row.empty:
        return selected_row['l_Script'].values[0]
    else:
        print('ERROR. No Exist Script')
        return None

In [11]:
def preprocessing_train_input(file_path: str, file_name: str, dictionary: Dict[str, str],index: int,input_df: str) -> Tuple[
    np.ndarray, np.ndarray, np.ndarray]:
    logger.debug(file_path)
    # text
    text = find_script_by_id(input_df, input_id = file_name)
    if text is None:
        logger.debug("text is None")
        return (None,None, None, None )

    frame_num = int(get_number_of_frames(file_path))
    #Audio
    extracted_audio_raw = extract_audio_waveform(file_path=file_path)#오디오 데이터 가져오기
    # preprocessed_audio = preprocess_audio_series(raw_data=extracted_audio_raw)#오디오 파일 처리
    # Video
    full_video_frames = extract_train_N_video_frames(frame_num=frame_num,file_path=file_path,index=index, number_of_samples=number_of_samples)#128프레임 획득
    resized_Full_images = [resize_image(image=im, new_size=(224, 224)) for im in full_video_frames]#크기를 조정하다
    # cropped_images = [crop_image_window(image=resi, training=training) / 255.0 for resi in resized_images]#对图像进行裁剪
    if len(full_video_frames)==number_of_samples:
        preprocessed_full_video = np.stack(resized_Full_images)#하나의 배열을 합성하다
        # Ground Truth
        video_gt = reading_label_data(file_name=file_name, dictionary=dictionary)#태그 읽기
        del full_video_frames
        return (extracted_audio_raw,preprocessed_full_video, text, video_gt)
    else:
        logger.debug("frames not enough")
    return (None,None,None,None)


In [12]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    x2_list = []
    x3_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
        x2_list.append(dataset[i][2])
        x3_list.append(dataset[i][3])
    return (x0_list, np.stack(x1_list), np.stack(x2_list), np.stack(x3_list))

In [13]:
csv_train_input = pd.read_csv(filepath_or_buffer="/ssd_data/qx/preprocessing/csv/FI_train_pp.csv", encoding="ms932", sep=",")
csv_val_input = pd.read_csv(filepath_or_buffer="/ssd_data/qx/preprocessing/csv/FI_val_pp.csv", encoding="ms932", sep=",")

### extract train data(original_cartoon_fusion) rate based

In [ ]:
import os
import pickle
import datetime
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

# ===== 기존에 사용하던 의존 함수들 (자식 프로세스에서 import 가능해야 함) =====
# from your_module import preprocessing_train_input, get_number_of_frames, reshape_to_expected_input

# -------- 멀티프로세싱 worker 초기화: 읽기 전용 대용량 객체를 전역으로 등록하여 복사 비용 감소 --------
_GT = None
_BASE_PATH = None

def _init_worker(gt_dict, base_path):
    global _GT, _BASE_PATH
    _GT = gt_dict
    _BASE_PATH = base_path

# -------- 개별 파일 처리 로직 (각 자식 프로세스에서 실행) --------
def _process_one(filename):
    try:
        # 숨김 파일 또는 비정상 파일은 건너뜀
        if filename.startswith('.'):
            return None

        file_path = os.path.join(_BASE_PATH, filename)

        # 프레임 수 기반 필터링
        # (주의: 이 함수가 매우 느리다면 preprocessing 함수 내부로 이동하여 한 번만 계산하도록 최적화 가능)
        try:
            nframes = int(get_number_of_frames(file_path))
            if nframes <= number_of_samples + 1:
                return None
        except Exception:
            # 손상된 메타데이터 / 읽기 불가 파일은 바로 건너뜀
            return None

        # 기존의 사용자 정의 전처리 수행
        # index 는 병렬 처리에서는 의미 없으므로 None 전달 (또는 메인 프로세스에서 관리)
        data = preprocessing_train_input(file_path=file_path,
                                         file_name=filename,
                                         dictionary=_GT,
                                         index=None, input_df=csv_val_input)

        # 전처리 실패 또는 비정상 데이터는 제외
        if not data or data[0] is None:
            return None
        return data

    except Exception:
        # 어떤 예외가 발생하더라도 해당 샘플을 건너뛰어 전체 실행이 중단되지 않도록 처리
        return None


if __name__ == "__main__":
    # ====== 경로 및 데이터 설정 ======
    allsavename = '/ssd_data/qx/Personality_Llama/fullshot_rowaudio_label_120frames/valid_set.dat'
    base_path   = '/ssd_data/qx/Big5/valid'
    gt_path     = "/ssd_data/qx/Big5/gt/annotation_validation.pkl"

    with open(gt_path, "rb") as f:
        gt = pickle.load(f, encoding='latin1')

    # ====== 처리 시간 측정 시작 ======
    t1 = datetime.datetime.utcnow()

    # 처리 대상 파일 목록 구성
    file_list = [fn for fn in os.listdir(base_path) if not fn.startswith('.')]
    total = len(file_list)

    # 사용할 프로세스 수 지정
    # 시스템 안정성을 위해 1개 코어는 예약하고, I/O 중심 작업이면 개수를 더 늘릴 수 있음
    workers = max(1, min(cpu_count() - 1, 16))

    # 적절한 chunksize 설정은 프로세스 간 통신 오버헤드를 크게 줄여줌
    chunksz = 8  

    training_set_data = []

    with Pool(processes=workers,
              initializer=_init_worker,
              initargs=(gt, base_path)) as pool:

        # imap_unordered: 결과가 준비되는 대로 바로 소비하여 메모리 사용량이 적고, tqdm 으로 진행률 표시 가능
        for res in tqdm(pool.imap_unordered(_process_one, file_list, chunksize=chunksz),
                        total=total,
                        desc=f"Building valid_set with {workers} workers",
                        unit="file"):
            if res is not None:
                training_set_data.append(res)

    # ====== 처리된 결과를 한 번에 저장 ======
    with open(allsavename, "wb") as f:
        pickle.dump(reshape_to_expected_input(training_set_data), f, protocol=pickle.HIGHEST_PROTOCOL)

    t2 = datetime.datetime.utcnow()
    print(f"Processed {len(training_set_data)}/{total} files.")
    print('Elapsed time: ' + str(t2 - t1))


Building valid_set with 16 workers:   0%|          | 0/2000 [00:00<?, ?file/s]

ERROR. No Exist Script


Building valid_set with 16 workers:   6%|▌         | 121/2000 [04:32<14:53,  2.10file/s]   

ERROR. No Exist Script


Building valid_set with 16 workers:  12%|█▏        | 249/2000 [08:33<16:11,  1.80file/s]  

...

ERROR. No Exist Script


Building valid_set with 16 workers:  95%|█████████▌| 1905/2000 [1:01:59<02:30,  1.58s/file]

ERROR. No Exist Script
ERROR. No Exist Script
ERROR. No Exist Script


Building valid_set with 16 workers: 100%|██████████| 2000/2000 [1:03:02<00:00,  1.89s/file]


Processed 1970/2000 files.
Elapsed time: 1:03:28.861898


In [ ]:
import os
import pickle
import datetime
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

# ===== 기존 함수는 반드시 모듈 최상단에서 import 가능해야 함 =====
# from your_module import preprocessing_train_input, get_number_of_frames, reshape_to_expected_input

# ---------- 프로세스 간 공유되는 읽기 전용 전역 객체 ----------
_GT = None
_BASE_PATH = None

def _init_worker(gt_dict, base_path):
    """자식 프로세스 초기화: 큰 읽기 전용 객체를 전역 변수에 저장해 불필요한 복사를 줄임."""
    global _GT, _BASE_PATH
    _GT = gt_dict
    _BASE_PATH = base_path

def _process_one(filename):
    """단일 파일 처리: 프레임 수 필터링 → 전처리 → 데이터 반환 또는 None"""
    try:
        if not filename or filename.startswith('.'):
            return None

        file_path = os.path.join(_BASE_PATH, filename)

        # 프레임 수 필터링 (전처리 함수 내부에서 한 번만 영상 열도록 구조 변경 가능)
        try:
            nframes = int(get_number_of_frames(file_path))
            if nframes <= number_of_samples + 1:
                return None
        except Exception:
            return None

        data = preprocessing_train_input(
            file_path=file_path,
            file_name=filename,
            dictionary=_GT,
            index=None, input_df=csv_train_input  # 병렬 처리에서는 index가 로그 외 의미 없으므로 전달 불필요
        )
        if not data or data[0] is None:
            return None
        return data

    except Exception:
        return None


if __name__ == "__main__":
    # ====== 경로 및 메타데이터 ======
    allsavename = '/ssd_data/qx/Personality_Llama/fullshot_rowaudio_label_120frames/train_set.dat'
    base_path   = '/ssd_data/qx/Big5/train'
    gt_path     = "/ssd_data/qx/Big5/gt/annotation_training.pkl"

    with open(gt_path, "rb") as f:
        gt = pickle.load(f, encoding='latin1')

    # ====== 시간 측정 ======
    t1 = datetime.datetime.utcnow()

    # 처리할 파일 목록
    file_list = [fn for fn in os.listdir(base_path) if not fn.startswith('.')]
    total = len(file_list)

    # 프로세스 수 및 청크 크기 설정
    workers = max(1, min(cpu_count() - 1, 16))  # 시스템을 위해 1코어는 남겨둠; 필요에 따라 조정 가능
    chunksz = 8  # 적절한 chunksize는 프로세스 간 통신 비용을 크게 줄임

    training_set_data = []

    with Pool(processes=workers, initializer=_init_worker, initargs=(gt, base_path)) as pool:
        for res in tqdm(pool.imap_unordered(_process_one, file_list, chunksize=chunksz),
                        total=total,
                        desc=f"Building train_set with {workers} workers",
                        unit="file"):
            if res is not None:
                training_set_data.append(res)

    # ====== 데이터 저장 ======
    with open(allsavename, "wb") as f:
        pickle.dump(reshape_to_expected_input(training_set_data), f, protocol=pickle.HIGHEST_PROTOCOL)

    t2 = datetime.datetime.utcnow()
    print(f"Processed {len(training_set_data)}/{total} files.")
    print('Elapsed time: ' + str(t2 - t1))


Building train_set with 16 workers:   2%|▏         | 129/6000 [06:41<8:15:25,  5.06s/file] 

ERROR. No Exist Script


Building train_set with 16 workers:   3%|▎         | 177/6000 [07:45<2:07:47,  1.32s/file]

ERROR. No Exist Script


Building train_set with 16 workers:   4%|▍         | 249/6000 [09:06<2:26:27,  1.53s/file]

...

ERROR. No Exist Script


Building train_set with 16 workers:  99%|█████████▉| 5945/6000 [3:03:49<01:04,  1.17s/file]

ERROR. No Exist Script


Building train_set with 16 workers: 100%|██████████| 6000/6000 [3:04:19<00:00,  1.84s/file]


Processed 5896/6000 files.
Elapsed time: 3:07:40.091972
